# RAG(Retrieval Augmented Generation) use local model, production grade RAG.


### 1. import the libraries and load the data

In [1]:
import os
import re
from dotenv import load_dotenv
from pymupdf import pymupdf
load_dotenv()
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
import chromadb
import pymupdf4llm


In [ ]:
def process_all_pdfs(data):
    """Process all PDF files in the specified directory."""
    all_documents = []
    pdf_dir = Path(data)

    # Find all PDF files in the directory and store all the docs in a list can use repeatedly.
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing file:{pdf_file}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            
            all_documents.extend(documents)
            print(f"\n Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"\n Error: {e}")

    print(f"\n Total documents loaded:{len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("data")

Found 1 PDF files to process.

Processing file:data\Claude Certified Architect – Foundations Certification Exam Guide.pdf

 Loaded 40 pages

 Total documents loaded:40


In [5]:
doc = pymupdf.open("data/Claude Certified Architect – Foundations Certification Exam Guide.pdf")

for page in doc:
    for block in page.get_text("dict")["blocks"]:
        if "lines" in block:
            for line in block["lines"]:
                for span in line["spans"]:
                    print(span["size"],span["text"][:60])

        

12.0  
16.0 Claude Certified Architect – Foundations Certification Exam 
14.0 Introduction 
12.0 The 
12.0 Claude Certified Architect – Foundations
12.0  certification validates that practitioners can make 
12.0 informed decisions about tradeoffs when implementing real-wo
12.0 exam tests foundational knowledge across Claude Code, the Cl
12.0 and Model Context Protocol (MCP) — the core technologies use
12.0 applications with Claude. 
12.0  
12.0 Questions on this exam are grounded in realistic scenarios d
12.0 cases, including building agentic systems for customer suppo
12.0 pipelines, integrating Claude Code into CI/CD workflows, bui
12.0 and extracting structured data from unstructured documents. 
12.0 only conceptual knowledge but practical judgment about archi
12.0 tradeoffs in production deployments. 
12.0  
12.0 This guide describes the exam content, lists the domains and
12.0 sample questions, and recommends preparation strategies. Use
12.0 experience to prepare effectively. 
14.

In [3]:
all_pdf_documents

[Document(metadata={'source': 'data\\Claude Certified Architect – Foundations Certification Exam Guide.pdf', 'file_path': 'data\\Claude Certified Architect – Foundations Certification Exam Guide.pdf', 'page': 0, 'total_pages': 40, 'format': 'PDF 1.4', 'title': 'Claude Certified Architect – Foundations Certification Exam Guide', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'Skia/PDF m148 Google Docs Renderer', 'creationDate': '', 'modDate': '', 'trapped': '', 'source_file': 'Claude Certified Architect – Foundations Certification Exam Guide.pdf', 'file_type': 'pdf'}, page_content=' \nClaude Certified Architect – Foundations Certification Exam Guide \nIntroduction \nThe Claude Certified Architect – Foundations certification validates that practitioners can make \ninformed decisions about tradeoffs when implementing real-world solutions with Claude. This \nexam tests foundational knowledge across Claude Code, the Claude Agent SDK, the Claude API, \nand Model Cont

In [4]:
# Remove headers and footers from the page content and merge pages of the same PDF file to one document.
Boilerplate_patterns = [
    re.compile(r"^\s*ANTHROPIC\s*\n",re.IGNORECASE),
    re.compile(r"Anthropic,\s*PBC\s*·\s*Confidential Need to Know\s*\(NTK\)",re.IGNORECASE),
]

def clean_page(text):
    """Clean the page content by removing boilerplate text."""
    text = text.replace("\f", "")
    for pattern in Boilerplate_patterns:
        text = pattern.sub("", text)
    # Normalise whitespace-only blank lines: \n  \n to \n\n to fix page breaks issue.
    text = re.sub(r"\n[ \t]+\n", "\n\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def merge_pages_per_file(documents):
    """Merge pages of the same PDF file into a single document to solve the chunking not overplap between pages."""
    per_file = {} #group documents per each file so not merge all pdfs into one document
    for doc in documents:
        key = doc.metadata.get("source")
        per_file.setdefault(key, []).append(doc)

    merged_docs = []
    
    for source_file, docs in per_file.items():
        # join with double newline so paragraphs splitting still works
        full_text = "\n\n".join(clean_page(p.page_content) for p in docs)
        merged_docs.append(Document(
            page_content=full_text,
            metadata={"source_file": source_file, "file_type": "pdf"}
        ))

    return merged_docs    


In [5]:
merged_pdf_docs = merge_pages_per_file(all_pdf_documents)
print(f"Total merged documents: {len(merged_pdf_docs)}")
print(f"First merged document content:\n{merged_pdf_docs[0].page_content[:2300]}")

Total merged documents: 1
First merged document content:
Claude Certified Architect – Foundations Certification Exam Guide 
Introduction 
The Claude Certified Architect – Foundations certification validates that practitioners can make 
informed decisions about tradeoffs when implementing real-world solutions with Claude. This 
exam tests foundational knowledge across Claude Code, the Claude Agent SDK, the Claude API, 
and Model Context Protocol (MCP) — the core technologies used to build production-grade 
applications with Claude. 

Questions on this exam are grounded in realistic scenarios drawn from actual customer use 
cases, including building agentic systems for customer support, designing multi-agent research 
pipelines, integrating Claude Code into CI/CD workflows, building developer productivity tools, 
and extracting structured data from unstructured documents. Candidates must demonstrate not 
only conceptual knowledge but practical judgment about architecture, configuration, 

## 2. Chunking strategies 
### fixed size, recursive text splitter, section chunk, sentence chunk, semantic chunk

In [6]:
# Chunking the documents using RecursiveCharacterTextSplitter from Langchain, it is a smarter variant of fixed_size chunking.It tries to cut at natural boundaries
def chunk_by_recursive_char(merged_pdf_docs, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=[ "\n", ". "," ", ""] #dropped "\n\n" because normalised blank lines to \n\n, the chunks wouldnot overlap if split at \n\n
    )
    split_docs = text_splitter.split_documents(merged_pdf_docs)

    return split_docs

In [7]:
recursive_char = chunk_by_recursive_char(merged_pdf_docs)
print(f"Split {len(merged_pdf_docs)} documents into {len(recursive_char)} chunks with recursive character splitting")

for i, chunk in enumerate(recursive_char[:5]):
    print(f"--- Chunk {i} ---")
    print(f"Content: {chunk.page_content}\n")
    print(f"Metadata: {chunk.metadata}\n")
    

Split 1 documents into 98 chunks with recursive character splitting
--- Chunk 0 ---
Content: Claude Certified Architect – Foundations Certification Exam Guide 
Introduction 
The Claude Certified Architect – Foundations certification validates that practitioners can make 
informed decisions about tradeoffs when implementing real-world solutions with Claude. This 
exam tests foundational knowledge across Claude Code, the Claude Agent SDK, the Claude API, 
and Model Context Protocol (MCP) — the core technologies used to build production-grade 
applications with Claude. 

Questions on this exam are grounded in realistic scenarios drawn from actual customer use 
cases, including building agentic systems for customer support, designing multi-agent research 
pipelines, integrating Claude Code into CI/CD workflows, building developer productivity tools, 
and extracting structured data from unstructured documents. Candidates must demonstrate not 
only conceptual knowledge but practical judgment

In [8]:
#Overlap test - chunk 2&3 doesnot have overlap on previous page, see if any chunk 2 tail reappearing in chunk 3.
chunk_a = recursive_char[2].page_content
chunk_b = recursive_char[3].page_content

for window in [200, 150, 100, 50, 30]:
    tail = chunk_a[-window:]
    if tail in chunk_b[:400]:
        print(f"Overlap of {window} chars confirmed.")
        print(f"Overlapping text: {tail}")
        break
    else:
        print(f"No overlap detected with wondow size")

print(f"Chunk 0 length: {len(recursive_char[0].page_content)}")
print(f"Chunk 1 length: {len(recursive_char[1].page_content)}")
print(f"Chunk 2 length: {len(chunk_a)}")
print(f"Chunk 3 length: {len(chunk_b)}")



No overlap detected with wondow size
Overlap of 150 chars confirmed.
Overlapping text: Code, and MCP, understanding both the capabilities and limitations of large 
language models in production environments. 
Exam Content 
Response Types
Chunk 0 length: 984
Chunk 1 length: 957
Chunk 2 length: 922
Chunk 3 length: 955


In [9]:
# Fixed-size chunking by character count with overlap. This way the chunks may not make sense because it cuts in the middle of sentences.
def chunks_by_char(merged_pdf_docs, chunk_size=1000, chunk_overlap=200):
    chunks = []

    for doc in merged_pdf_docs:
            page = doc.page_content
            start_idx = 0

            while start_idx < len(page):
                end_idx = min(start_idx + chunk_size, len(page))
                chunk = page[start_idx:end_idx]
                chunks.append(chunk)

                start_idx = (
                end_idx - chunk_overlap if end_idx < len(page) else end_idx
             )

    return chunks   

In [10]:
all_pdf_chunks = chunks_by_char(merged_pdf_docs)
print(f"Split {len(merged_pdf_docs)} documents into {len(all_pdf_chunks)} character chunks with overlap")
for i, chunk in enumerate(all_pdf_chunks[:5]):
    print(f"--- Chunk{i} ---\n{chunk}\n")

Split 1 documents into 97 character chunks with overlap
--- Chunk0 ---
Claude Certified Architect – Foundations Certification Exam Guide 
Introduction 
The Claude Certified Architect – Foundations certification validates that practitioners can make 
informed decisions about tradeoffs when implementing real-world solutions with Claude. This 
exam tests foundational knowledge across Claude Code, the Claude Agent SDK, the Claude API, 
and Model Context Protocol (MCP) — the core technologies used to build production-grade 
applications with Claude. 

Questions on this exam are grounded in realistic scenarios drawn from actual customer use 
cases, including building agentic systems for customer support, designing multi-agent research 
pipelines, integrating Claude Code into CI/CD workflows, building developer productivity tools, 
and extracting structured data from unstructured documents. Candidates must demonstrate not 
only conceptual knowledge but practical judgment about architecture, c

In [11]:
def chunks_by_sentences(merged_pdf_docs, max_sentences_per_chunk=5, overlap_sentences=1):
    chunks = []

    for doc in merged_pdf_docs:
        page = doc.page_content
        sentences = re.split(r"(?<=[.?!])\s+",page)

        start_idx = 0
        while start_idx < len(sentences):
            end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
            current_chunk = sentences[start_idx:end_idx]
            chunks.append(" ".join(current_chunk))
            start_idx = end_idx - overlap_sentences if end_idx < len(sentences) else end_idx

    return chunks


In [12]:
all_pdf_sentence_chunks = chunks_by_sentences(merged_pdf_docs)
print(f"Split {len(merged_pdf_docs)} documents into {len(all_pdf_sentence_chunks)} sentence chunks")
for i, chunk in enumerate(all_pdf_sentence_chunks[:5]):
    print(f"--- Sentence Chunk {i} ---\n{chunk}\n")

Split 1 documents into 56 sentence chunks
--- Sentence Chunk 0 ---
Claude Certified Architect – Foundations Certification Exam Guide 
Introduction 
The Claude Certified Architect – Foundations certification validates that practitioners can make 
informed decisions about tradeoffs when implementing real-world solutions with Claude. This 
exam tests foundational knowledge across Claude Code, the Claude Agent SDK, the Claude API, 
and Model Context Protocol (MCP) — the core technologies used to build production-grade 
applications with Claude. Questions on this exam are grounded in realistic scenarios drawn from actual customer use 
cases, including building agentic systems for customer support, designing multi-agent research 
pipelines, integrating Claude Code into CI/CD workflows, building developer productivity tools, 
and extracting structured data from unstructured documents. Candidates must demonstrate not 
only conceptual knowledge but practical judgment about architecture, configu

##### For chunking, will proceed with langchain recursive text splitter becasue this is the best option for general-purpose default for PDF. (need to remove the page break to combine the pages if the content span to different pages to make sure overlap happens. Dropped of paragraph split \n\n because no small enough trailing split for overlap after combine the pages. )
##### Fixed size with overlap splitter has hard stop and break the words which causing missing info/lack context.
##### Chunk by sentences leave the chunk sizes varies in different size which is not consistent for token ceiling.
##### Chunk by section in PDF is chunking by pages not section - it works the best on markdown files.
##### Will try semantic chunking later which is more expensive because use LLM to evaluate.


## 3. Embeddings & Vector Database

In [13]:
def create_embeddings(documents):
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    if Path("./chroma_db").exists(): # load existing vector store
        vector_store = Chroma(collection_name="pdf_chunks", embedding_function=embedding_model, persist_directory="./chroma_db")
    else:
        vector_store = Chroma.from_documents(recursive_char, embedding_model, collection_name="pdf_chunks",persist_directory="./chroma_db")
    return vector_store

In [14]:
vectorstore=create_embeddings(recursive_char)

In [15]:
print(vectorstore._collection.count())

98


### 4. Eval test set with similarity_search as baseline metrics

In [36]:
# eval test with 15-20 sample query to see if the vector store is working
query1 = "What content domains and weightings do the exam have?"
results = vectorstore.similarity_search(query1, k=5)
for i, result in enumerate(results):
    print(f"\n--- Result {i} ---\n")
    print(result.page_content)
    print(result.metadata)


--- Result 0 ---

Scaled scoring models help equate scores across multiple exam forms that might have slightly 
different difficulty levels. 
Content Outline 
This exam guide includes weightings, content domains, and task statements for the exam. 

The exam has the following content domains and weightings: 

-​
Domain 1: Agentic Architecture & Orchestration (27% of scored content) 
-​
Domain 2: Tool Design & MCP Integration (18% of scored content)

-​
Domain 3: Claude Code Configuration & Workflows (20% of scored content) 
-​
Domain 4: Prompt Engineering & Structured Output (20% of scored content) 
-​
Domain 5: Context Management & Reliability (15% of scored content) 

 
Exam Scenarios 
The exam uses scenario-based questions. Each scenario presents a realistic production context 
that frames a set of questions. During the exam, 4 scenarios will  be presented and picked at 
random from the full set of the 6 scenarios below. 
Scenario 1: Customer Support Resolution Agent
{'file_type': '

Query 1 retrieval came back with Result 0 which is correct.

In [40]:
# query 2 to test the vector store
query2 = "What hands-on experience the idea candidate with for the exam?"
results = vectorstore.similarity_search(query2, k=10)
for i, result in enumerate(results):
    print(f"\n--- Result {i} ---\n")
    print(result.page_content)



--- Result 0 ---

fields to prevent hallucination 
-​
Few-shot prompting: Ambiguous scenario targeting, format consistency, false positive 
reduction 
-​
Batch processing: Message Batches API appropriateness, latency tolerance assessment, 
failure handling by custom_id 
-​
Context window optimization: Trimming verbose tool outputs, structured fact extraction, 
position-aware input ordering 
-​
Human review workflows: Confidence calibration, stratified sampling, accuracy 
segmentation by document type and field 
-​
Information provenance: Claim-source mappings, temporal data handling, conflict 
annotation, coverage gap reporting 
Out-of-Scope Topics 
The following related topics will NOT appear on the exam: 

-​
Fine-tuning Claude models or training custom models 
-​
Claude API authentication, billing, or account management 
-​
Detailed implementation of specific programming languages or frameworks (beyond 
what's needed for tool and schema configuration) 
-​

--- Result 1 ---

Scaled 

Retrieve results for query 2: Answers in Result 8 and 9 when increase to K=10 from k=5, do I need to rerank or BM25?

In [41]:
# test with query 3
query3 = "What are out of scope topics?"
results = vectorstore.similarity_search(query3, k=15)
for i, result in enumerate(results):
    print(f"\n--- Result {i} ---\n")
    print(result.page_content)



--- Result 0 ---

In-Scope Topics 
The following topics are explicitly tested on the exam: 

-​
Agentic loop implementation: Control flow based on stop_reason, tool result handling, 
loop termination conditions 
-​
Multi-agent orchestration: Coordinator-subagent patterns, task decomposition, parallel 
subagent execution, iterative refinement loops 
-​
Subagent context management: Explicit context passing, structured state persistence, 
crash recovery using manifests 
-​
Tool interface design: Writing effective tool descriptions, splitting vs consolidating tools, 
tool naming to reduce ambiguity 
-​
MCP tool and resource design: Resources for content catalogs, tools for actions, 
description quality for adoption 
-​
MCP server configuration: Project vs user scope, environment variable expansion, 
multi-server simultaneous access 
-​
Error handling and propagation: Structured error responses, transient vs business vs 
permission errors, local recovery before escalation 
-​

--- Result 1

Retrieval result for query 3: increased k=5 to 15 because only found result 1 only has part of the answers and even k=15 still not retrieve the other half. Next will try BM25 and rerun if this improves. And then hybrid or reranker.

In [ ]:
query4 = "What is the response type in this exam?"
results = vectorstore.similarity_search(query4, k=5)
for i, result in enumerate(results):
    print(f"\n --- Result {i} ---\n")
    print(result.page_content)
    


 --- Result 0 ---

-​
Choosing hooks over prompt-based enforcement when business rules require guaranteed 
compliance 
Task Statement 1.6: Design task decomposition strategies for complex workflows 
Knowledge of: 

-​
When to use fixed sequential pipelines (prompt chaining) versus dynamic adaptive 
decomposition based on intermediate findings 
-​
Prompt chaining patterns that break reviews into sequential steps (e.g., analyze each file 
individually, then run a cross-file integration pass) 
-​
The value of adaptive investigation plans that generate subtasks based on what is 
discovered at each step 

Skills in: 

-​
Selecting task decomposition patterns appropriate to the workflow: prompt chaining for 
predictable multi-aspect reviews, dynamic decomposition for open-ended investigation 
tasks 
-​
Splitting large code reviews into per-file local analysis passes plus a separate cross-file 
integration pass to avoid attention dilution 
-​

 --- Result 1 ---

reports, weekly audits, night

Query 4: result 0 is covered the answer.

In [42]:
query5 = "What will be the exam result looks like?"
results = vectorstore.similarity_search(query5, k=5)
for i, results in enumerate(results):
    print(f"\n --- Result {i} ---\n")
    print(results.page_content)




 --- Result 0 ---

SDK, Claude Code, and MCP, understanding both the capabilities and limitations of large 
language models in production environments. 
Exam Content 
Response Types 
All questions on the exam are multiple choice format. Each question has one correct response 
and three incorrect responses (distractors). 

Select the single response that best completes the statement or answers the question. 
Distractors are response options that a candidate with incomplete knowledge or experience 
might choose. 

Unanswered questions are scored as incorrect; there is no penalty for guessing. 
Exam Results 
The exam has a pass or fail designation. The exam is scored against a minimum standard 
established by subject matter experts. 

Your results are reported as a scaled score of 100–1,000. The minimum passing score is 720. 
Scaled scoring models help equate scores across multiple exam forms that might have slightly 
different difficulty levels. 
Content Outline

 --- Result 1 ---

Scal

Query 5 retrieved result 0 which is correct

In [43]:
query6 = "If I got 710, did I pass the exam?"
results = vectorstore.similarity_search(query6, k=5)
for i, results in enumerate(results):
    print(f"\n --- Result {i} ---\n")
    print(results.page_content)


 --- Result 0 ---

SDK, Claude Code, and MCP, understanding both the capabilities and limitations of large 
language models in production environments. 
Exam Content 
Response Types 
All questions on the exam are multiple choice format. Each question has one correct response 
and three incorrect responses (distractors). 

Select the single response that best completes the statement or answers the question. 
Distractors are response options that a candidate with incomplete knowledge or experience 
might choose. 

Unanswered questions are scored as incorrect; there is no penalty for guessing. 
Exam Results 
The exam has a pass or fail designation. The exam is scored against a minimum standard 
established by subject matter experts. 

Your results are reported as a scaled score of 100–1,000. The minimum passing score is 720. 
Scaled scoring models help equate scores across multiple exam forms that might have slightly 
different difficulty levels. 
Content Outline

 --- Result 1 ---

Scal

query 6 retrieved Result 0 that cover the score. Is it correct to my question?

In [45]:
query7 = "What topics are NOT covered in this exam?"
results = vectorstore.similarity_search(query7, k=5)
for i, results in enumerate(results):
    print(f"\n --- Result {i} ---\n")
    print(results.page_content)


 --- Result 0 ---

Scaled scoring models help equate scores across multiple exam forms that might have slightly 
different difficulty levels. 
Content Outline 
This exam guide includes weightings, content domains, and task statements for the exam. 

The exam has the following content domains and weightings: 

-​
Domain 1: Agentic Architecture & Orchestration (27% of scored content) 
-​
Domain 2: Tool Design & MCP Integration (18% of scored content)

-​
Domain 3: Claude Code Configuration & Workflows (20% of scored content) 
-​
Domain 4: Prompt Engineering & Structured Output (20% of scored content) 
-​
Domain 5: Context Management & Reliability (15% of scored content) 

 
Exam Scenarios 
The exam uses scenario-based questions. Each scenario presents a realistic production context 
that frames a set of questions. During the exam, 4 scenarios will  be presented and picked at 
random from the full set of the 6 scenarios below. 
Scenario 1: Customer Support Resolution Agent

 --- Result 1

In [53]:
query8 = "How long will the exam takes?"
results = vectorstore.similarity_search_with_score(query8, k=5)
for i, (doc, score) in enumerate(results):
    print(f"\n --- Result {i} ---\n")
    print(f"Score: {score}\n")
    print(doc.page_content)


 --- Result 0 ---

Score: 1.0158525705337524

SDK, Claude Code, and MCP, understanding both the capabilities and limitations of large 
language models in production environments. 
Exam Content 
Response Types 
All questions on the exam are multiple choice format. Each question has one correct response 
and three incorrect responses (distractors). 

Select the single response that best completes the statement or answers the question. 
Distractors are response options that a candidate with incomplete knowledge or experience 
might choose. 

Unanswered questions are scored as incorrect; there is no penalty for guessing. 
Exam Results 
The exam has a pass or fail designation. The exam is scored against a minimum standard 
established by subject matter experts. 

Your results are reported as a scaled score of 100–1,000. The minimum passing score is 720. 
Scaled scoring models help equate scores across multiple exam forms that might have slightly 
different difficulty levels. 
Content Outli

In [54]:
query9 = "What would happen if I used Plan mode instead of direct execution?"
results = vectorstore.similarity_search_with_score(query9, k=5)
for i, (doc, score) in enumerate(results):
    print(f"\n --- Result {i} ---\n")
    print(f"Score: {score}\n")
    print(doc.page_content)



 --- Result 0 ---

Score: 1.0603585243225098

implementation approach before making changes. B) Start with direct execution and make 
changes incrementally, letting the implementation reveal the natural service boundaries. C) Use 
direct execution with comprehensive upfront instructions detailing exactly how each service 
should be structured. D) Begin in direct execution mode and only switch to plan mode if you 
encounter unexpected complexity during implementation. 

Correct Answer: A 

Plan mode is designed for complex tasks involving large-scale changes, multiple valid 
approaches, and architectural decisions—exactly what monolith-to-microservices restructuring 
requires. It enables safe codebase exploration and design before committing to changes. Option 
B risks costly rework when dependencies are discovered late. Option C assumes you already 
know the right structure without exploring the code. Option D ignores that the complexity is 
already stated in the requirements, not som

In [48]:
query10 = "What are the workflows to reduce API costs for automated analysis."
results = vectorstore.similarity_search(query10, k=5)
for i, results in enumerate(results):
    print(f"\n --- Result {i} ---\n")
    print(results.page_content)


 --- Result 0 ---

-​
Choosing hooks over prompt-based enforcement when business rules require guaranteed 
compliance 
Task Statement 1.6: Design task decomposition strategies for complex workflows 
Knowledge of: 

-​
When to use fixed sequential pipelines (prompt chaining) versus dynamic adaptive 
decomposition based on intermediate findings 
-​
Prompt chaining patterns that break reviews into sequential steps (e.g., analyze each file 
individually, then run a cross-file integration pass) 
-​
The value of adaptive investigation plans that generate subtasks based on what is 
discovered at each step 

Skills in: 

-​
Selecting task decomposition patterns appropriate to the workflow: prompt chaining for 
predictable multi-aspect reviews, dynamic decomposition for open-ended investigation 
tasks 
-​
Splitting large code reviews into per-file local analysis passes plus a separate cross-file 
integration pass to avoid attention dilution 
-​

 --- Result 1 ---

reports, weekly audits, night

Result 2 for query 10